# Grad-CAM Analysis Per Task Head

Post-hoc only: it inspects the chosen model and informs no selection or tuning decision.

The instance is the **median** of the three seeds by joint accuracy on the **validation** set -- median rather than best so it reflects typical behaviour, validation rather than test so the test set stays out of every selection decision. Images are a stratified random sample fixed in advance: 2 per class combination, 48 in total.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'
CHECKPOINT_DIR = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/fish-freshness-mtl/results'
FIGURES_DIR = f'{RESULTS_DIR}/figures/gradcam'

import os
for d in ['/content/data', FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)
!unzip -q -n "$DATASET_ZIP" -d /content/data

## Choose the instance

Set `BEST_MODEL` to the multi-task variant that came out of notebook 05; the seed is then picked automatically.

In [ ]:
import pandas as pd
import torch

from evaluate import load_multitask_model
from gradcam_utils import select_median_seed, sample_images
from split_utils import load_split

BEST_MODEL = 'ModelD_UW'  # set from notebook 05
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

val_joint = pd.read_csv(f'{RESULTS_DIR}/validation_joint_accuracy.csv')
candidates = val_joint[val_joint.model == BEST_MODEL].set_index('seed')['val_joint_accuracy'].to_dict()
SELECTED_SEED = select_median_seed(candidates)

print('validation joint accuracy per seed:', {k: round(v, 4) for k, v in candidates.items()})
print('median seed selected:', SELECTED_SEED)

RUN_NAME = f'{BEST_MODEL}_seed{SELECTED_SEED}'
model = load_multitask_model(f'{CHECKPOINT_DIR}/{RUN_NAME}.pt', DEVICE)

In [ ]:
train_df, val_df, test_df = load_split('/content/repo/02_Manifests/split_manifest.csv')
sample_df = sample_images(test_df)
print(f'{len(sample_df)} images across {sample_df.combined_class.nunique()} combinations')
sample_df[['species', 'freshness', 'filepath']].head()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

from dataset import eval_transform
from gradcam_utils import compute_gradcam, denormalize_for_display

for i, row in sample_df.iterrows():
    pil_image = Image.open(os.path.join(DATASET_ROOT, row['filepath'])).convert('RGB')
    img_tensor = eval_transform(pil_image)
    rgb_float = denormalize_for_display(img_tensor)

    cam_species = compute_gradcam(model, img_tensor, rgb_float, head='species')
    cam_freshness = compute_gradcam(model, img_tensor, rgb_float, head='freshness')

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(rgb_float); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(cam_species); axes[1].set_title('Species Head'); axes[1].axis('off')
    axes[2].imshow(cam_freshness); axes[2].set_title('Freshness Head'); axes[2].axis('off')
    fig.suptitle(f"{row['species']} - {row['freshness']}  [{RUN_NAME}]")
    plt.tight_layout()

    safe = f"{row['species']}_{row['freshness']}_{i:02d}".replace(' ', '_')
    fig.savefig(os.path.join(FIGURES_DIR, f'{safe}.png'), dpi=150)
    plt.show(); plt.close(fig)

In [ ]:
with open(f'{RESULTS_DIR}/gradcam_provenance.txt', 'w') as f:
    f.write(f'model: {BEST_MODEL}\nseed: {SELECTED_SEED}\n')
    f.write(f'selection: median validation joint accuracy of {candidates}\n')
    f.write(f'images: {len(sample_df)} stratified random, 2 per combination\n')
print('provenance recorded so the analysis can be repeated exactly')